[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/cookbook/blob/main/gen-ai/agents/ecommerce-agent/ecommerce-agent.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/aurelio-labs/cookbook/blob/main/gen-ai/agents/ecommerce-agent/ecommerce-agent.ipynb)

# Building an Shopping Copilot with KumoRFM

In this example we will use KumoRFM to power an ecommerce platform shopping assistant.

In [ ]:
!pip install -qU \
    "datasets>=4.0.0" \
    "graphai-lib==0.0.10rc3" \
    "ipykernel>=6.30.1" \
    "ipywidgets>=8.1.7" \
    "kumoai==2.7.0" \
    "openai>=1.99.9"

## Building the Ecommerce Prediction Engine

We'll be using KumoRFM to power our shopping predictions. To make calls to KumoRFM an API key is required, you can a free API key via the widget below:

In [1]:
import os
from kumoai.experimental import rfm

if not os.environ.get("KUMO_API_KEY"):
    rfm.authenticate()

Opening browser page to automatically generate an API key...


[2025-09-12 07:22:23 - kumoai:298 - INFO] Generated token "sdk-jamess-macbook-pro.local-2025-09-12-07-22-15-Z" and saved to KUMO_API_KEY env variable


In [2]:
rfm.init(api_key=os.environ["KUMO_API_KEY"])

[2025-09-12 07:22:33 - kumoai:203 - INFO] Successfully initialized the Kumo SDK against deployment https://kumorfm.ai/api, with log level INFO.


We're going to use a sample of the H&M ecommerce dataset. The sample is available on Hugging Face Datasets at [jamescalam/hm-sample](https://huggingface.co/datasets/jamescalam/hm-sample). It includes three tables, the `customers` table with 1.1K rows, `articles` with 5K rows, and `transactions` with 15.7K rows.

In [3]:
from datasets import load_dataset

customers = load_dataset(
    "jamescalam/hm-sample", data_files="customers.jsonl", split="train"
)
customers

Dataset({
    features: ['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code'],
    num_rows: 1100
})

In [4]:
articles = load_dataset(
    "jamescalam/hm-sample", data_files="articles.jsonl", split="train"
)
articles

Dataset({
    features: ['article_id', 'product_code', 'prod_name', 'product_type_no', 'product_type_name', 'product_group_name', 'graphical_appearance_no', 'graphical_appearance_name', 'colour_group_code', 'colour_group_name', 'perceived_colour_value_id', 'perceived_colour_value_name', 'perceived_colour_master_id', 'perceived_colour_master_name', 'department_no', 'department_name', 'index_code', 'index_name', 'index_group_no', 'index_group_name', 'section_no', 'section_name', 'garment_group_no', 'garment_group_name', 'detail_desc'],
    num_rows: 5000
})

In [5]:
transactions = load_dataset(
    "jamescalam/hm-sample", data_files="transactions.jsonl", split="train"
)
transactions

Dataset({
    features: ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id'],
    num_rows: 15773
})

We'll read these into Kumo by first transforming our _data**sets**_ into Pandas _data**frames**_:

In [6]:
customers_df = customers.to_pandas()
articles_df = articles.to_pandas()
transactions_df = transactions.to_pandas()

Once we have our dataframes we will transform them into `rfm.LocalTable` objects. These are lightweight abstractions of pandas dataframes that allow us to interface our data with KumoRFM. We use the `.infer_metadata()` method to automatically infer what types of data we have in our tables:

In [7]:
customers = rfm.LocalTable(customers_df, name="customers").infer_metadata()
transactions = rfm.LocalTable(transactions_df, name="transactions").infer_metadata()
articles = rfm.LocalTable(articles_df, name="articles").infer_metadata()

Detected primary key 'customer_id' in table 'customers'
Detected time column 't_dat' in table 'transactions'
Detected primary key 'article_id' in table 'articles'


We can update the column types as needed like so:

In [8]:
# update semantic type of columns
customers["customer_id"].stype = "ID"
customers["age"].stype = "numerical"

# primary keys
customers.primary_key = "customer_id"
articles.primary_key = "article_id"

# time column
transactions.time_column = "t_dat"

Then we create the graph:

In [9]:
# select the tables
graph = rfm.LocalGraph(tables=[
    customers, transactions, articles
])
# link the tables
graph.link(src_table="transactions", fkey="customer_id", dst_table="customers")
graph.link(src_table="transactions", fkey="article_id", dst_table="articles")

LocalGraph(
  tables=[
    customers,
    transactions,
    articles,
  ],
  edges=[
    transactions.customer_id ⇔ customers.customer_id,
    transactions.article_id ⇔ articles.article_id,
  ],
)

Now we setup our KumoRFM model on our graph:

In [10]:
model = rfm.KumoRFM(graph=graph)

Output()

Now we can make predictions, let's see how likely one of our products are to be purchased over the next 30 days:

In [11]:
article_id = articles_df.iloc[0].article_id.item()
article_id

675662003

In [12]:
# forecast 30-day product demand for specific item/article
df = model.predict(
    f"PREDICT SUM(transactions.price, 0, 30, days) FOR articles.article_id={article_id}"
)
display(df)

Output()

,ENTITY,ANCHOR_TIMESTAMP,TARGET_PRED
0,675662003,1600732800000,0.000009


We have a pretty low likelihood of that product being purchased over the next 30 days. Let's see how likely two of our customers are to be purchases something over the next 90 days.

In [13]:
csample = customers_df.iloc[:2].customer_id.tolist()
csample

['1935b6baf9d28d1f19b7ffad18a9da418954a9bf38f59336f2f86d7a5615d1d2',
 '75ebdc56559b1f2739ce5832bd85a921ba827c72383135bdcc08a616d320e948']

In [14]:
# predict likelihood of two specific users not ordering in the next 90 days
df = model.predict(
    "PREDICT COUNT(transactions.*, 0, 90, days)=0 "
    f"FOR customers.customer_id IN ('{csample[0]}', '{csample[1]}')"
)
display(df)

Output()

,ENTITY,ANCHOR_TIMESTAMP,TARGET_PRED,False_PROB,True_PROB
0,1935b6baf9d28d1f19b7ffad18a9da418954a9bf38f593...,1600732800000,False,0.672468,0.327532
1,75ebdc56559b1f2739ce5832bd85a921ba827c72383135...,1600732800000,False,0.746204,0.253796


Now the next step is to build an AI agent that can do this for us.

## Building the Agent

In [17]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") or \
    getpass("Enter your OpenAI API key: ")

We then generate completions like so:

In [18]:
from openai import AsyncOpenAI

client = AsyncOpenAI()

response = await client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Tell me something interesting about GNNs"}
    ],
    stream=True,
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

Certainly! One interesting fact about Graph Neural Networks (GNNs) is that they excel at capturing relationships and interactions in data that is naturally represented as graphs, such as social networks, molecules, or citation networks. Unlike traditional neural networks, which often work on fixed-size, Euclidean data (like images or sequences), GNNs can process variable-sized, non-Euclidean data structures.

A particularly fascinating aspect of GNNs is how they use a message-passing framework: each node in the graph updates its representation by aggregating information (messages) from its neighbors. This iterative process allows GNNs to learn complex dependencies in the graph topology, making them highly effective for tasks like node classification, link prediction, and even modeling physical systems.

Moreover, recent advances in GNNs have led to models that are provably as powerful as the Weisfeiler-Lehman graph isomorphism test in distinguishing different graph structures, highligh

Now we setup our agent with tools. We use the "no-framework" framework [GraphAI](https://docs.aurelio.ai/graphai/get-started/introduction).

Using this library we are expected to create our own tool functions, LLM API calls, etc. The library primarily acts as a graph execution framework _without_ any AI abstractions. With that in mind we will first define _two_ tools for our agent.

### Tool 1: Query Dataframes

The first tool runs a namespace `exec` instance allowing our LLM to run python code against our pandas dataframes.

With `graphai` we typically define tools with two components, a pydantic `BaseModel` to outline the tool schema for our agent, and the python function that will be executed when the tool is called.

In [19]:
import json
import pandas as pd
from pydantic import BaseModel, Field
from graphai import node
from graphai.callback import EventCallback


class QueryDataframes(BaseModel):
    """Execute simple filtered queries on the ecommerce dataframes. Will execute code in
    a namespace with the following dataframes:
    
    - transactions_df
    - articles_df
    - customers_df
    
    You can also access pandas library via `pd` for dataframe operations. Ensure you use
    assign the results you need to the `out` variable, otherwise nothing will be returned
    as this will be run with `exec()`. After execution we access the `out` variable and
    return it to you.

    If outputting a dataframe, you must use the .to_markdown() method to output an easily
    readable markdown table.
    """
    query: str = Field(..., description="The python code to execute")

@node(stream=True)
async def query_dataframes(input: dict, state: dict, callback: EventCallback) -> dict:
    try:
        tool_call_args = json.loads(state["events"][-1]["tool_calls"][0]["function"]["arguments"])
        # get dataframes, pandas, and set `out` to None
        namespace = {
            "transactions_df": state["transactions_df"],
            "articles_df": state["articles_df"],
            "customers_df": state["customers_df"],
            "pd": pd,
            "out": None,
        }
        # grab query from LLM to be executed
        query = tool_call_args.get("query")
        if not query:
            raise ValueError("No query provided")
        # remove escaped newlines as it frequently breaks the query
        query = query.replace("\\n", "\n")
        # execute query within predefined namespace
        exec(query, namespace)
        # pull out the `out` value
        out = namespace.get("out")
        if out is None:
            out = "No result returned via the `out` variable"
        content = [{"type": "text", "text": json.dumps(out, default=str)}]
    except Exception as e:
        content = [{
            "type": "text",
            "text": (
                f"Error executing query: {str(e)}. "
                "Please fix your query and trying again."
            )
        }]
    # stream tool output
    await callback.acall(
        type="tool_output",
        params={
            "id": state["events"][-1]["tool_calls"][0]["id"],
            "name": "predict_customer_purchase",
            "arguments": tool_call_args,
            "output": content[0]["text"]
        }
    )
    # Add tool call event to state
    event = {
        "role": "tool",
        "content": content,
        "tool_call_id": state["events"][-1]["tool_calls"][0]["id"]
    }
    state["events"].append(event)
    return {"input": {}}

[2025-09-12 07:24:16 - graphai.utils:160 - WARNING] Function query_dataframes has no docstring


### Tool 2: Query KumoRFM

The second tool will provide access to KumoRFM's PQL queries. For this tool to work, we need to add some guidelines on how to use it for our agent. We'll first grab those, the full prompt used can be [found here](https://github.com/jamescalam/ecommerce-agent/blob/main/api/pluto/prompts/developer.py).

In [20]:
import requests

pql_file = requests.get(
    "https://raw.githubusercontent.com/jamescalam/ecommerce-agent/refs/heads/main/api/pluto/prompts/developer.py"
).text
# strip first and last two lines as they contain python boilerplate
pql_reference = "\n".join(pql_file.split("\n")[1:-2])
print(pql_reference[:200])

# KumoRFM Predictive Query Language (PQL) Reference

## Overview

Predictive Query Language (PQL) is KumoRFM's declarative SQL-like syntax for defining predictive modeling tasks using the foundation m


Given the size of these guidelines we'll insert them directly into our system/developer message rather than our tool description.

Now we define the KumoRFM tool like so:

In [21]:
class KumoRFM(BaseModel):
    """This tool allows you to write any PQL query to the KumoRFM model.
    """
    query: str = Field(..., description="The PQL query to predict")

@node(stream=True)
async def kumorfm(input: dict, state: dict, callback: EventCallback) -> dict:
    try:
        tool_call_args = json.loads(state["events"][-1]["tool_calls"][0]["function"]["arguments"])
        query = tool_call_args.get("query")
        if not query:
            raise ValueError("No query provided")
        
        df = state["kumorfm"].predict(query)
        out = df.to_dict(orient="records")
        content = [{"type": "text", "text": json.dumps(out)}]
    except Exception as e:
        content = [{"type": "text", "text": str(e)}]
    # stream tool output
    await callback.acall(
        type="tool_output",
        params={
            "id": state["events"][-1]["tool_calls"][0]["id"],
            "name": "predict_customer_purchase",
            "arguments": tool_call_args,
            "output": content[0]["text"]
        }
    )
    event = {
        "role": "tool",
        "content": content,
        "tool_call_id": state["events"][-1]["tool_calls"][0]["id"]
    }
    state["events"].append(event)
    return {"input": {}}

[2025-09-12 07:24:35 - graphai.utils:160 - WARNING] Function kumorfm has no docstring


For our LLM to be able to read our tool schemas, we will be using the built-in `FunctionSchema` method. We can use this to consume our pydantic base models and later output them into an OpenAI-friendly schema format.

In [22]:
from graphai.utils import FunctionSchema

query_df_schema = FunctionSchema.from_pydantic(QueryDataframes)
query_df_schema.name = "query_dataframes"
kumorfm_schema = FunctionSchema.from_pydantic(KumoRFM)
kumorfm_schema.name = "kumorfm"

tools = [query_df_schema, kumorfm_schema]

The schemas can then be created using the `to_openai` method (when using OpenAI models).

In [23]:
tools[1].to_openai(api="completions")

{'type': 'function',
 'function': {'name': 'kumorfm',
  'description': 'This tool allows you to write any PQL query to the KumoRFM model.\n    ',
  'parameters': {'type': 'object',
   'properties': {'query': {'description': 'The PQL query to predict',
     'type': 'string'}},
   'required': ['query']}}}

### Building the Graph

Graphs are constructed from nodes and edges, with various special nodes and edges within that broader structure. For our use-case we don't need to dive into anything too exotic. All we need to do is define our nodes, and construct our graph to join those together.

#### Our Nodes

The graph will consist of _five_ total nodes, two of those we have already defined with our tools. The remaining three are:

- `llm` router node will contain the logic for calling our LLM and handling our LLM's tool-calling decisions.
- `start` and `end` nodes are `graphai`-specific boilerplate, they act as the entry and exit points of our graph

We will first define the `llm` router:

In [24]:
from graphai import router

@router(stream=True)
async def llm(input: dict, state: dict, callback: EventCallback) -> dict:
    # get client initialized in lifespan
    client = state["client"]
    # call openai (or another provider as preferred)
    stream = await client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=state["events"],
        tools=[x.to_openai(api="completions") for x in tools],
        stream=True,
        seed=9000,  # keep consistent results
        parallel_tool_calls=False,
    )
    direct_answer: str = ""
    tool_call: dict = {}
    tool_call_args = ""
    async for chunk in stream:
        if (token := chunk.choices[0].delta.content) is not None:
            # this handles direct text output
            direct_answer += token
            await callback.acall(token=token)
        # handle tool calls
        tool_calls_out = chunk.choices[0].delta.tool_calls
        if tool_calls_out and (tool_name := tool_calls_out[0].function.name) is not None:
            # this handles the initial tokens of a tool call
            tool_call["id"] = tool_calls_out[0].id
            tool_call["name"] = tool_name
            # we can return the tool name
            await callback.acall(
                type="tool_call",
                params=tool_call
            )
        elif tool_calls_out and (tool_args := tool_calls_out[0].function.arguments) is not None:
            # this handles the arguments of a tool call
            tool_call_args += tool_args
            # we can output these too
            await callback.acall(
                type="tool_args",
                params={
                    **tool_call,
                    "arguments": tool_args
                }
            )
    if direct_answer:
        # if we got a direct answer we create a standard assistant message
        state["events"].append(
            {
                "role": "assistant",
                "content": direct_answer,
            }
        )
        # choice controls the next node destination
        choice = "end"
    elif tool_call:
        # if we got a tool call we create an assistant tool call message
        state["events"].append(
            {
                "role": "assistant",
                "tool_calls": [{
                    "id": tool_call["id"],
                    "type": "function",
                    "function": {
                        "name": tool_call["name"],
                        "arguments": tool_call_args,
                    }
                }]
            }
        )
        choice = tool_call["name"]
    return {"input": input, "choice": choice}

[2025-09-12 07:24:41 - graphai.utils:160 - WARNING] Function llm has no docstring


And now our two boilerplate `start` and `end` nodes:

In [25]:
@node(start=True)
async def start(input: dict) -> dict:
    return {"input": input}

@node(end=True)
async def end(input: dict, state: dict) -> dict:
    return {"output": state["events"]}

[2025-09-12 07:24:42 - graphai.utils:160 - WARNING] Function start has no docstring
[2025-09-12 07:24:42 - graphai.utils:160 - WARNING] Function end has no docstring


#### Constructing the Graph

Our broader graph contains the logic that connects our various nodes and defines the initial state of the workflow. We will first define our state, which will consist of our initial system/developer message, our KumoRFM instance, and the three H&M dataframes.

We will begin by defining the developer message:

In [48]:
dev_message = {
    "role": "developer",
    "content": (
        "You are a helpful assistant that uses the various tools and "
        "KumoRFM integration to answer the user's analytics questions "
        "about our H&M ecommerce dataset."
        "\n"
        "When answering questions, you may use the various tools "
        "multiple times before answering to the user. You should aim "
        "aim to have all of the information you need from the tools "
        "before answering the user."
        "\n"
        "There is a limit of 30 steps to each interaction, measured "
        "as the number of tool calls made between the user's most "
        "recent message and your response to the user. Keep that limit "
        "in mind but ensure you are still thorough in your analysis."
        # remember to include our KumoRFM reference guide
        "\n\n"
        "## PQL (Predictive Query Language) Reference\n"
        "Use this syntax when working with KumoRFM predictions:\n"
        "\n"
        f"{pql_reference}"
    )
}

And now our initial state:

In [49]:
initial_state = {
    "events": [dev_message],
    "kumorfm": model,
    "transactions_df": transactions_df,
    "articles_df": articles_df,
    "customers_df": customers_df,
    "client": client
}

This state can be added to our graph using the `set_state` method. Alongside this we will also be adding the various nodes and routers to our graph with `add_node` and `add_router`. We then set all edges with `add_edge`. Finally, once our graph is fully defined we `compile` it.

In [50]:
from graphai import Graph

# create graph
graph = (
    Graph(max_steps=30)
    .set_state(initial_state)
    .add_node(start)
    .add_node(llm)
    .add_node(kumorfm)
    .add_node(query_dataframes)
    .add_node(end)
    .add_router(
        sources=[start],
        router=llm,
        destinations=[
            kumorfm,
            query_dataframes,
            end
        ]
    )
    .add_edge(kumorfm, llm)
    .add_edge(query_dataframes, llm)
    .add_edge(llm, end)
    .compile()
)

## Using our Agent

The agent is now fully defined and we can start using it. We call it with `await graph.execute` like so:

In [51]:
import asyncio

cb = EventCallback()
# add our input message to the state
graph.update_state({
    "events": [
        *graph.state["events"],
        {
            "role": "user",
            "content": f"Can you predict the demand for article {article_id} over the next 30 days"
        }
    ]
})
# now execute
_ = asyncio.create_task(
    graph.execute({"input": {}}, callback=cb)
)

# and (optionally) stream the output
async for event in cb.aiter():
    if str(event.type) == "callback":
        # this indicates direct text output
        print(event.token, end="", flush=True)
    elif event.type == "tool_call":
        # this indicates the first event in a tool call
        # this contains tool name and ID
        print(event.params["name"], flush=True)
    elif event.type == "tool_args":
        # this indicates the arguments of a tool call
        print(event.params["arguments"], end="", flush=True)
    elif event.type == "tool_output":
        # this indicates the output of a tool call
        # these can be very long so we'll avoid printing them
        # but feel free to try
        #print(event.params["output"])
        print()
        pass

kumorfm
{"query":"PREDICT SUM(transactions.price, 0, 30, days)\nFOR articles.article_id IN ('675662003')"}
kumorfm
{"query":"PREDICT SUM(transactions.price, 0, 30, days)\nFOR articles.article_id = '675662003'"}
kumorfm
{"query":"PREDICT SUM(transactions.price, 0, 30, days)\nFOR articles.article_id = 675662003"}

Output()


The predicted demand (sum of sales price) for article 675662003 over the next 30 days is approximately 9.21e-06, which suggests extremely low or negligible demand expected for this article in the upcoming month.

Let's write a helper function for our chat:

In [52]:
async def chat(content: str):
    cb = EventCallback()
    graph.update_state({
        "events": [
            *graph.state["events"],
            {"role": "user", "content": content}
        ]
    })

    _ = asyncio.create_task(
        graph.execute({"input": {}}, callback=cb)
    )
    
    async for event in cb.aiter():
        if str(event.type) == "callback":
            # this handles direct text output
            print(event.token, end="", flush=True)
        elif event.type == "tool_call":
            # this indicates the first event in a tool call
            # this contains tool name and ID
            print(event.params["name"], flush=True)
        elif event.type == "tool_args":
            # this indicates the arguments of a tool call
            print(event.params["arguments"], end="", flush=True)
        elif event.type == "tool_output":
            # this indicates the output of a tool call
            # these can be very long so we'll avoid printing them
            # but feel free to try
            #print(event.params["output"])
            print()
            pass
        

In [53]:
await chat(
    "What other useful info can you give me? I'm preparing our monthly marketing "
    "emails"
)

query_dataframes
{"query":"# Let's look into some potentially useful insights for marketing emails\n# 1. Top selling articles in last 30 days\n# 2. Customer segments based on purchase behavior\n# 3. Popular categories\n\n# Top selling articles in last 30 days\ntop_selling_articles = transactions_df[transactions_df['t_dat'] > (transactions_df['t_dat'].max() - pd.Timedelta(days=30))]\ntop_selling_articles = top_selling_articles.groupby('article_id').agg({'price': 'sum'}).reset_index().sort_values(by='price', ascending=False).head(10)\n\n# Popular categories in last 30 days\npopular_categories = transactions_df[transactions_df['t_dat'] > (transactions_df['t_dat'].max() - pd.Timedelta(days=30))]\npopular_categories = popular_categories.merge(articles_df[['article_id', 'product_type_name']], on='article_id')\npopular_categories = popular_categories.groupby('product_type_name').agg({'price': 'sum'}).reset_index().sort_values(by='price', ascending=False).head(5)\n\n# Customer segments based o

In [54]:
await chat("Can you help me find customers likely to churn?")

kumorfm
{"query":"PREDICT COUNT(transactions.*, 0, 30, days) = 0\nFOR customers.customer_id IN (SELECT DISTINCT customer_id FROM transactions_df) \nWHERE COUNT(transactions.*, -90, 0, days) > 0"}
query_dataframes
{"query":"# Fetch all distinct customer_ids from transactions\ndistinct_customers = transactions_df['customer_id'].unique().tolist()\nout = distinct_customers[:50]  # Limiting to first 50 for prediction sample"}
kumorfm
{"query":"PREDICT COUNT(transactions.*, 0, 30, days) = 0\nFOR customers.customer_id IN ('8ef783d3815860cc145c2923f740f88728e373f2c3cb37aa638c15810ac531cc', '2d545e697d8cf36558c81eb56c1776cb30f893585ed21aa1531863c727a42fbb', '4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28', '26e237aa2bc47082d06d49af58bbd65785cb979daf3110313d1484b95adac609', '8df45859ccd71ef1e48e2ee9d1c65d5728c31c46ae957d659fa4e5c3af6cc076', '59470fe7e847d5c05976da6c41fd27fa221b1fb7f7e3b76d2509994011435375', 'd9d809b2a22dfe4afcbe5351c5c3ca2ac6f375ae0dba65156ec9ea422428053b', '01

Output()


I predicted churn likelihood for a sample of customers who have purchased in the last 90 days. Here are the results for six customers:

- Customer 8ef783d3815860cc145c2923f740f88728e373f2c3cb37aa638c15810ac531cc has about 70.9% probability to churn (not make a purchase) in next 30 days.
- Customer 2d545e697d8cf36558c81eb56c1776cb30f893585ed21aa1531863c727a42fbb has about 60.3% probability to churn.
- Customer 4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28 has about 74.9% probability to churn.
- Customer 26e237aa2bc47082d06d49af58bbd65785cb979daf3110313d1484b95adac609 has about 60.1% probability to churn.
- Customer 8df45859ccd71ef1e48e2ee9d1c65d5728c31c46ae957d659fa4e5c3af6cc076 has about 38.3% probability to churn.
- Customer 59470fe7e847d5c05976da6c41fd27fa221b1fb7f7e3b76d2509994011435375 has about 72.0% probability to churn.

I can continue and extract more customers likely to churn or analyze churn for specific segments if you want.

In [55]:
await chat("Can you get a sample of 50 customers?")

kumorfm
{"query":"PREDICT COUNT(transactions.*, 0, 30, days) = 0\nFOR customers.customer_id IN ('8ef783d3815860cc145c2923f740f88728e373f2c3cb37aa638c15810ac531cc', '2d545e697d8cf36558c81eb56c1776cb30f893585ed21aa1531863c727a42fbb', '4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28', '26e237aa2bc47082d06d49af58bbd65785cb979daf3110313d1484b95adac609', '8df45859ccd71ef1e48e2ee9d1c65d5728c31c46ae957d659fa4e5c3af6cc076', '59470fe7e847d5c05976da6c41fd27fa221b1fb7f7e3b76d2509994011435375', 'd9d809b2a22dfe4afcbe5351c5c3ca2ac6f375ae0dba65156ec9ea422428053b', '01c19c0ba392de6d2bee657a616eca254d8fa6d06dde299b73d4276381b54554', '03d0011487606c37c1b1ed147fc72f285a50c05f00b9712e0fc3da400c864296', 'c4e748d5bf4f10c86410d8b0cf62535ace6b502a80ed253ab4328f3eb3ca32ca', 'fc4842d6365813761635df4f175fcbea80f9a30d03366f7b9743b8ae18edd14b', '3b9dd61ce941502be31b2214187705adefc8ea3036ba61c3d3d07b74b10c1588', '423f107d7078f6fd4d32211a2a345934fa06ef1d711ddf2dfc99c5f2331c86f5', '129575cfdb9a72e7167

Output()


I have predicted the likelihood of churn (no purchase in next 30 days) for a sample of 50 customers who have made purchases in the last 90 days.

Out of these:
- Some customers have high probability (above 70%) of churning.
- Some have moderate probability (around 50-70%).
- A few have lower probability (below 50%).

If you want, I can provide you with the detailed list of these customers along with their churn probabilities for your marketing targeting. Would you like that?

In [56]:
await chat("Okay, but are those recently active customers?")

query_dataframes
{"query":"# Let's check the last purchase date of these 50 customers to see if they are recently active\nsample_customer_ids = ['8ef783d3815860cc145c2923f740f88728e373f2c3cb37aa638c15810ac531cc', '2d545e697d8cf36558c81eb56c1776cb30f893585ed21aa1531863c727a42fbb', '4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28', '26e237aa2bc47082d06d49af58bbd65785cb979daf3110313d1484b95adac609', '8df45859ccd71ef1e48e2ee9d1c65d5728c31c46ae957d659fa4e5c3af6cc076', '59470fe7e847d5c05976da6c41fd27fa221b1fb7f7e3b76d2509994011435375', 'd9d809b2a22dfe4afcbe5351c5c3ca2ac6f375ae0dba65156ec9ea422428053b', '01c19c0ba392de6d2bee657a616eca254d8fa6d06dde299b73d4276381b54554', '03d0011487606c37c1b1ed147fc72f285a50c05f00b9712e0fc3da400c864296', 'c4e748d5bf4f10c86410d8b0cf62535ace6b502a80ed253ab4328f3eb3ca32ca', 'fc4842d6365813761635df4f175fcbea80f9a30d03366f7b9743b8ae18edd14b', '3b9dd61ce941502be31b2214187705adefc8ea3036ba61c3d3d07b74b10c1588', '423f107d7078f6fd4d32211a2a345934fa06ef

In [57]:
await chat(
    "Okay let's use these, let's filter down to the most likely to churn who also have past "
    "purchase history with us"
)

kumorfm
{"query":"PREDICT COUNT(transactions.*, 0, 30, days) = 0\nFOR customers.customer_id IN ('8ef783d3815860cc145c2923f740f88728e373f2c3cb37aa638c15810ac531cc', '2d545e697d8cf36558c81eb56c1776cb30f893585ed21aa1531863c727a42fbb', '4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28', '26e237aa2bc47082d06d49af58bbd65785cb979daf3110313d1484b95adac609', '8df45859ccd71ef1e48e2ee9d1c65d5728c31c46ae957d659fa4e5c3af6cc076', '59470fe7e847d5c05976da6c41fd27fa221b1fb7f7e3b76d2509994011435375', 'd9d809b2a22dfe4afcbe5351c5c3ca2ac6f375ae0dba65156ec9ea422428053b', '01c19c0ba392de6d2bee657a616eca254d8fa6d06dde299b73d4276381b54554', '03d0011487606c37c1b1ed147fc72f285a50c05f00b9712e0fc3da400c864296', 'c4e748d5bf4f10c86410d8b0cf62535ace6b502a80ed253ab4328f3eb3ca32ca', 'fc4842d6365813761635df4f175fcbea80f9a30d03366f7b9743b8ae18edd14b', '3b9dd61ce941502be31b2214187705adefc8ea3036ba61c3d3d07b74b10c1588', '423f107d7078f6fd4d32211a2a345934fa06ef1d711ddf2dfc99c5f2331c86f5', '129575cfdb9a72e7167

Output()


query_dataframes
{"query":"# Filter these customers to those with at least one purchase in last 365 days\nsample_customer_ids = ['8ef783d3815860cc145c2923f740f88728e373f2c3cb37aa638c15810ac531cc', '2d545e697d8cf36558c81eb56c1776cb30f893585ed21aa1531863c727a42fbb', '4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28', '26e237aa2bc47082d06d49af58bbd65785cb979daf3110313d1484b95adac609', '8df45859ccd71ef1e48e2ee9d1c65d5728c31c46ae957d659fa4e5c3af6cc076', '59470fe7e847d5c05976da6c41fd27fa221b1fb7f7e3b76d2509994011435375', 'd9d809b2a22dfe4afcbe5351c5c3ca2ac6f375ae0dba65156ec9ea422428053b', '01c19c0ba392de6d2bee657a616eca254d8fa6d06dde299b73d4276381b54554', '03d0011487606c37c1b1ed147fc72f285a50c05f00b9712e0fc3da400c864296', 'c4e748d5bf4f10c86410d8b0cf62535ace6b502a80ed253ab4328f3eb3ca32ca', 'fc4842d6365813761635df4f175fcbea80f9a30d03366f7b9743b8ae18edd14b', '3b9dd61ce941502be31b2214187705adefc8ea3036ba61c3d3d07b74b10c1588', '423f107d7078f6fd4d32211a2a345934fa06ef1d711ddf2dfc99c

Output()


I filtered the sample of 50 customers to those who also have a purchase history in the past 365 days. Among these, the most likely to churn (not make a purchase in the next 30 days) have churn probabilities ranging up to about 83%.

Would you like me to provide the detailed list of these customers with their churn probabilities for your targeted marketing campaigns?

In [58]:
await chat("can you give me the top 3?")

Here are the top 3 customers most likely to churn (not make a purchase in next 30 days) among those who have purchased in the last 365 days:

1. Customer ID: 4ec3e4c6c1f78b6c8b381117394b3f1aca50a6910c71a5796be4a2eab7e1fc38  
   Churn Probability: 83.2%

2. Customer ID: 4330e0469755c75b92a58a5f5002c729479147d470e6cc42a3206572352a1e28  
   Churn Probability: 81.8%

3. Customer ID: 580869f7448924f37b6aea590b37f8183bef108bc3dbeee1117120a96c0f0712  
   Churn Probability: 81.3%

These customers are high priority targets for retention or re-engagement campaigns. Would you like assistance with specific marketing strategies or message ideas for them?

In [59]:
await chat(
    "let's write a personalized email to the first customer on that list - using what "
    "we know about their past purchases and predicted most likely future purchases"
)

query_dataframes
{"query":"# Fetch past purchase details for the customer\ncustomer_id = '4ec3e4c6c1f78b6c8b381117394b3f1aca50a6910c71a5796be4a2eab7e1fc38'\npast_purchases = transactions_df[transactions_df['customer_id'] == customer_id].merge(articles_df, on='article_id')\npast_purchases_summary = past_purchases.groupby('product_type_name').size().reset_index(name='count').sort_values(by='count', ascending=False)\n\n# Predict most likely future purchases (top 3) for the customer\nfuture_purchases_query = f\"PREDICT LIST_DISTINCT(transactions.article_id, 0, 30, days) RANK TOP 3 FOR customers.customer_id = '{customer_id}'\"\nfuture_purchases = None\nout = past_purchases_summary  # Outputing past purchases for now, prediction will be fetched next"}
kumorfm
{"query":"PREDICT LIST_DISTINCT(transactions.article_id, 0, 30, days) RANK TOP 3 FOR customers.customer_id = '4ec3e4c6c1f78b6c8b381117394b3f1aca50a6910c71a5796be4a2eab7e1fc38'"}

Output()


query_dataframes
{"query":"# Fetch article details for predicted future purchases\nfuture_article_ids = [703737001, 633208001, 659460002]\nfuture_articles = articles_df[articles_df['article_id'].isin(future_article_ids)][['article_id', 'prod_name', 'product_type_name']]\nout = future_articles"}
Based on the customer's past purchases, they have bought mostly Sweaters (4 times), Hoodies (2 times), Dress, Jacket, Scarf, and Underwear bottom.

The top 3 articles they are most likely to purchase in the next 30 days are:
1. PE - CLARA SCARF (Article ID: 703737001, Category: Scarf)
2. Love Lock Down (Article ID: 633208001, Category: Dress)
3. Joel light down jacket (Article ID: 659460002, Category: Jacket)

Here is a draft for a personalized marketing email for this customer:

---

Subject: Exclusive Picks Just for You - Cozy Up with Our Latest Collection!

Dear Valued Customer,

We noticed you love our cozy sweaters and stylish hoodies! To keep your wardrobe fresh and fabulous, we have hand

In [60]:
await chat("that feels a little too obviously automated, can you make it more natural?")

Certainly! Here is a more natural and friendly version of the personalized email:

---

Subject: Thought You’d Like These Cozy New Arrivals

Hi there,

We’ve noticed you’ve got a great eye for cozy sweaters and hoodies — we think you’ll love some of the latest arrivals we’ve picked out just for you.

Here are a few items we think could be perfect additions to your wardrobe:
- The PE - CLARA SCARF — a nice way to add a little extra warmth and style.
- Love Lock Down Dress — great for both casual days and nights out.
- Joel Light Down Jacket — perfect for staying comfortable and looking sharp as the cooler weather sets in.

We’d love to see you back soon—there might even be a little surprise waiting for you when you drop by.

Take care,  
The H&M Team

---

Would you like me to help you with similar emails for other customers?

In [61]:
await chat(
    "that's better, but let's be specific about what we think they'd like? "
    "We should keep the product names but add the article IDs in square brackets and "
    "an image of the product will appear in the email"
)

Of course! Here's a more personalized and specific version of the email including product names with article IDs in square brackets and a note about the product images:

---

Subject: Thought You’d Like These Cozy New Arrivals

Hi there,

Since you’ve shown a love for cozy sweaters and hoodies, we thought you’d be interested in these picks we think you'd really like:

- PE - CLARA SCARF [703737001] — A soft and stylish scarf that'll keep you warm and add a touch of elegance to any outfit. (You'll see an image of this beautiful scarf in the email!)

- Love Lock Down Dress [633208001] — Perfect for both casual days and nights out, this dress pairs well with your favorite sweaters and jackets.

- Joel Light Down Jacket [659460002] — Stay comfortably warm while looking sharp with this lightweight yet effective jacket.

We hope one (or all!) of these catches your eye. We’d love to welcome you back soon — and who knows, there might even be a little surprise waiting for you when you do.

Take

---